# Deep Learning 042 — The Convolution Operation

The lesson's worked example: a horizontal-edge filter on a 6×6 image gives
**−765** exactly on the edge rows and 0 everywhere else, while a vertical filter
on the **same image** gives **all zeros**. This notebook reproduces both.

| Part | What we check |
|---|---|
| A | convolution written out, step by step |
| B | the −765 feature map |
| C | the same image, a different filter, nothing found |
| D | colour: three channels, one filter, one output |
| E | shrinkage adds up |

In [ ]:
import numpy as np
np.set_printoptions(linewidth=120, suppress=True)

def convolve(img, k):
    kh, kw = k.shape
    out = np.zeros((img.shape[0] - kh + 1, img.shape[1] - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.sum(img[i:i + kh, j:j + kw] * k)   # multiply, then sum
    return out

## Part A — One position, by hand

In [ ]:
patch = np.array([[0, 0, 0], [0, 0, 0], [255, 255, 255]], dtype=float)
horizontal = np.array([[ 1,  1,  1],
                       [ 0,  0,  0],
                       [-1, -1, -1]], dtype=float)
print("patch:\n", patch)
print("filter:\n", horizontal)
print("\nelement-wise product:\n", patch * horizontal)
print("sum ->", (patch * horizontal).sum())
assert (patch * horizontal).sum() == -765

## Part B — The whole image

Top half dark, bottom half bright: one horizontal edge across the middle.

In [ ]:
img = np.zeros((6, 6))
img[3:, :] = 255
print("image:\n", img)

fmap = convolve(img, horizontal)
print("\nfeature map (4x4):\n", fmap)
print("\nmost negative value:", fmap.min(), "at rows", np.unique(np.where(fmap == fmap.min())[0]))
assert fmap.min() == -765 and set(fmap.flatten()) == {0.0, -765.0}

The filter fired exactly where the intensity changed, and said nothing anywhere
else. **An edge is a change in intensity**, so detecting edges is detecting
numerical change.

## Part C — The same image, a vertical filter

In [ ]:
vertical = np.array([[1, 0, -1],
                     [1, 0, -1],
                     [1, 0, -1]], dtype=float)
vmap = convolve(img, vertical)
print("feature map:\n", vmap)
print("\nall zeros?", np.all(vmap == 0))
assert np.all(vmap == 0)
print("The image did not change. The question did.")

## Part D — Colour is three channels, and nothing else changes

In [ ]:
colour = np.stack([img, img, img], axis=-1)          # 6x6x3
k3 = np.stack([horizontal] * 3, axis=-1)             # 3x3x3

out = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        out[i, j] = np.sum(colour[i:i+3, j:j+3, :] * k3)

print("input shape", colour.shape, "filter shape", k3.shape, "-> output shape", out.shape)
print("peak:", out.min(), "= 3 x -765")
assert out.min() == -765 * 3
print("\nThe filter gains depth; the output stays 2-D. One filter, one feature map.")

## Part E — Shrinkage adds up

In [ ]:
n, f = 28, 3
print(f"{'layer':>6}{'size':>8}")
size = n
for layer in range(1, 7):
    size = size - f + 1
    print(f"{layer:>6}{size:>8}")
print(f"\n{n}x{n} becomes {size}x{size} after 6 layers - without padding, depth costs you pixels.")
assert size == 16

## What to take away

- Convolution is: **place the filter, multiply element-wise, sum, slide, repeat**.
- The output is a **feature map** saying where that pattern occurs.
- The horizontal filter gave **−765** on the edge rows and 0 elsewhere; the
  vertical filter gave **all zeros** on the same image.
- **Colour just adds depth to the filter**; one filter still gives one map.
- Each layer shrinks the map by `f − 1`, which is what padding fixes next.

## Exercises

1. Flip the image (bright on top). What sign does the edge response take, and why?
2. Build a diagonal filter and find the image it responds to.
3. Convolve with `np.ones((3,3))/9`. What has that filter done, and what is it
   usually called?